In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "9b4218e10a3d2063d07f15244ed04a398d4ec957"
assert (len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH"), "Pin the reviewed pushed commit before Colab validation"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
RUN_VERSION = "v1_panderm_base_c1_finetune"
ARCH = "panderm_base_vit_b16"
CHECKPOINT_FORMAT = "panderm_full_model_v1"
VARIANT = "C1"
DF_TARGET_COUNT = 585
VALIDATION_EPOCHS = 5
BATCH_SIZE = 16
ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = 128
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 10
LAYER_DECAY = 0.65
DROP_PATH = 0.2
PANDERM_UPSTREAM_REPO = "https://github.com/SiyuanYan1/PanDerm"
PANDERM_UPSTREAM_COMMIT = "fd7a80748ba7fc3e203fed88f909f4689d0d6f24"
PANDERM_CHECKPOINT_FILENAME = "panderm_bb_data6_checkpoint-499.pth"
PANDERM_CHECKPOINT_DRIVE_ID = "removed-from-public-history"
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-panderm-runs")
COCA_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
V1_ROOT = SHARED_RUN_ROOT / RUN_VERSION
VALIDATION_RUNS_ROOT = V1_ROOT / "validation_runs"
VALIDATION_RECORD = V1_ROOT / "validation_record.json"
LATEST_FAILURE_RECORD = V1_ROOT / "latest_validation_failure.json"
FORMAL_ROOT = V1_ROOT / "formal"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".panderm_shared_root.json"
CODE_DIR = Path("/content/panderm-code")
UPSTREAM_DIR = Path("/content/panderm-upstream")
WEIGHTS_CACHE = Path("/content/panderm-weights")
LOCAL_DATA_DIR = Path("/content/ham10000-data")

# PanDerm-Base C1 full fine-tuning validation

Run all validates `v1_panderm_base_c1_finetune` and never starts the three formal runs.
Test is unreachable in this notebook. Existing CoCa / DDPM / ResNet artifacts and the
public deployment stay read-only.

Weights are used under CC BY-NC-ND 4.0 for non-commercial academic research only. Adapted
(fine-tuned) weights must never be shared or deployed. Results are exploratory only.

## Phase 0 CHECK - Drive, shared root, pinned clones, dependencies, isolation guards

In [ ]:
import hashlib, json, os, base64, subprocess, sys, time
from google.colab import drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut; do not create a private replacement: {SHARED_RUN_ROOT}"
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
subprocess.run(["nvidia-smi"], check=True)
token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access to this repo is required (enable notebook access for accounts A/B/C)"
GH_TOKEN_PRESENT = True
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status, "clone must be a clean detached checkout of the pinned commit"
assert "@" not in remote and "x-access-token" not in remote, "clone URL must not embed a credential"
assert not UPSTREAM_DIR.exists(), f"fresh runtime required: {UPSTREAM_DIR}"
subprocess.run(["git", "clone", "--filter=blob:none", PANDERM_UPSTREAM_REPO, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM_DIR), "checkout", "--detach", PANDERM_UPSTREAM_COMMIT], check=True)
upstream_commit = subprocess.check_output(["git", "-C", str(UPSTREAM_DIR), "rev-parse", "HEAD"], text=True).strip()
upstream_status = subprocess.check_output(["git", "-C", str(UPSTREAM_DIR), "status", "--short"], text=True).strip()
assert upstream_commit == PANDERM_UPSTREAM_COMMIT and not upstream_status, "PanDerm upstream must be a clean detached checkout of the pinned commit"
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TORCH_HOME"] = "/content/torch-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==0.9.16", "gdown>=5.1", "pandas>=2.0", "pillow>=9.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from importlib.metadata import version
import torch
from ddpm_derm import panderm_run
assert torch.cuda.is_available() and version("timm") == "0.9.16"
assert panderm_run.UPSTREAM_COMMIT == PANDERM_UPSTREAM_COMMIT == upstream_commit
assert panderm_run.RUN_VERSION == RUN_VERSION and panderm_run.ARCH == ARCH
resolved_root = panderm_run.require_existing_shared_root(SHARED_RUN_ROOT)
drive_probe = panderm_run.probe_shared_drive(resolved_root)
sentinel = panderm_run.create_or_validate_sentinel(SHARED_ROOT_SENTINEL, shortcut_alias="ddpm-derm-panderm-runs", resolved_path=str(resolved_root), drive_folder_id=None, run_version=RUN_VERSION)
assert sentinel["shortcut_alias"] == "ddpm-derm-panderm-runs" and sentinel["resolved_path"] == str(resolved_root)
assert SHARED_RUN_ROOT != COCA_RUN_ROOT and COCA_RUN_ROOT not in SHARED_RUN_ROOT.parents, "PanDerm output root must be fully isolated from the CoCa runs"
coca_guard_paths = sorted(COCA_RUN_ROOT.rglob("validation_record.json")) + sorted(COCA_RUN_ROOT.rglob("_COMPLETED.json")) if COCA_RUN_ROOT.is_dir() else []
project_guard_paths = [SHARED_PROJECT_DIR / name for name in ("README.md", "HANDOFF.md", "EXPERIMENT_LOG.md")]
guard_paths = [path for path in coca_guard_paths + project_guard_paths if path.is_file()]
before_guard = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
panderm_run.require_no_deployment_contamination(CODE_DIR)
if V1_ROOT.exists():
    assert not VALIDATION_RECORD.exists(), f"a validation_record already exists; do not re-draw validation for the same version: {VALIDATION_RECORD}"
    assert not LATEST_FAILURE_RECORD.exists(), f"a prior gate already failed this version: {LATEST_FAILURE_RECORD}"
    assert not FORMAL_ROOT.exists(), f"formal artifacts must not exist before validation: {FORMAL_ROOT}"
panderm_run.ensure_tree(SHARED_RUN_ROOT, V1_ROOT.relative_to(SHARED_RUN_ROOT))
panderm_run.ensure_tree(SHARED_RUN_ROOT, VALIDATION_RUNS_ROOT.relative_to(SHARED_RUN_ROOT))
print(json.dumps({"commit": commit, "upstream_commit": upstream_commit, "drive_probe": drive_probe, "guard_files": len(guard_paths)}, indent=2))

## Phase 0 CHECK - official checkpoint download, SHA-256 pinning and license/provenance identity

In [ ]:
WEIGHTS_CACHE.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = WEIGHTS_CACHE / PANDERM_CHECKPOINT_FILENAME
if not CHECKPOINT_PATH.is_file():
    import gdown
    gdown.download(id=PANDERM_CHECKPOINT_DRIVE_ID, output=str(CHECKPOINT_PATH), quiet=False)
assert CHECKPOINT_PATH.is_file(), f"PanDerm checkpoint download failed: {CHECKPOINT_PATH}"
observed_checkpoint_sha256 = sha256(CHECKPOINT_PATH)
print("checkpoint bytes:", CHECKPOINT_PATH.stat().st_size)
print("checkpoint sha256:", observed_checkpoint_sha256)
print("expected (pinned):", panderm_run.EXPECTED_CHECKPOINT_SHA256)
# Upstream publishes no digest, so the first download is trust-on-first-use and MUST fail
# loud here until a human reviews the printed digest, pins it in panderm_run.py, pushes and
# re-pins EXPECTED_GIT_COMMIT. require_checkpoint_sha256 never accepts the placeholder.
checkpoint_sha256 = panderm_run.require_checkpoint_sha256(CHECKPOINT_PATH)
provenance = panderm_run.summarize_provenance()
assert provenance["license_review"]["license"] == "CC-BY-NC-ND-4.0"
assert provenance["license_review"]["finetuning_allowed"] is True
assert provenance["license_review"]["deployment_allowed"] is False
assert provenance["license_review"]["sharing_adapted_weights_allowed"] is False
assert provenance["contamination_review"]["image_level_ham10000_overlap"] == "not_independently_excludable"
assert provenance["contamination_review"]["independent_audit_possible"] is False
assert provenance["contamination_review"]["exact_fixed_validation_test_overlap"] == "unproven"
assert provenance["contamination_review"]["patient_level_overlap"] == "not_excludable"
assert provenance["contamination_review"]["ham10000_in_upstream_finetuning_or_evaluation"] == "yes_evaluation_benchmark"
assert provenance["contamination_review"]["loaded_checkpoint_is_pretraining_only"] is True
assert provenance["claim_boundary"] == "suggestive_exploratory_only"
assert provenance["deployment_allowed"] is False
provenance_clearance = panderm_run.require_provenance_clearance(upstream_commit=upstream_commit, checkpoint_sha256=checkpoint_sha256, purpose=panderm_run.VALIDATION_ONLY)
print(json.dumps(provenance_clearance, indent=2))

## Phase 1 CHECK - fixed split counts, image_id/lesion_id leakage and C1 construction

In [ ]:
import pandas as pd
staging_report = panderm_run.stage_validation_data(SHARED_PROJECT_DIR / "data", LOCAL_DATA_DIR)
assert staging_report["manifest_files_copied"] == 2
assert staging_report["class_mapping_files_copied"] == 1
assert staging_report["images_copied"] == 6995 + 1510
assert staging_report["test_manifest_present"] is False
assert staging_report["image_set_exact"] is True
os.environ["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
from ddpm_derm import config, manifests, classifier_objective
frames = {split: manifests.load_split(split) for split in ("train", "val")}
assert len(frames["train"]) == 6995 and len(frames["val"]) == 1510
assert int((frames["train"]["dx"] == "df").sum()) == 85 and int((frames["val"]["dx"] == "df").sum()) == 14
assert not set(frames["train"]["lesion_id"]) & set(frames["val"]["lesion_id"])
assert not set(frames["train"]["image_id"]) & set(frames["val"]["image_id"])
# PanDerm v1 identity is train/validation only; the test manifest is prohibited.
manifest_sha256 = {split: sha256(LOCAL_DATA_DIR / "manifests" / f"{split}.csv") for split in ("train", "val")}
fixed_split_identity = manifest_sha256["train"]
c1_frame = manifests.build_classifier_frame("C1", df_target_count=DF_TARGET_COUNT, seed=0)
c1_counts = classifier_objective.ordered_class_counts(c1_frame)
assert len(c1_frame) == panderm_run.EXPECTED_C1_TRAIN_ROWS == 7495
assert c1_counts == panderm_run.EXPECTED_C1_CLASS_COUNTS
assert c1_counts["df"] == DF_TARGET_COUNT
assert "source" not in c1_frame.columns or set(c1_frame["source"].unique()) <= {"real"}
real_df_ids = set(frames["train"].loc[frames["train"]["dx"] == "df", "image_id"].astype(str))
c1_df_ids = set(c1_frame.loc[c1_frame["dx"] == "df", "image_id"].astype(str))
assert c1_df_ids == real_df_ids, "C1 df rows must be duplicates of real train df images only"
assert not c1_df_ids & set(frames["val"]["image_id"].astype(str))
print(json.dumps({"c1_rows": len(c1_frame), "c1_counts": c1_counts, "manifest_sha256": manifest_sha256}, indent=2))

## Phase 2 CHECK - targeted unit / static / smoke checks from the pinned checkout

In [ ]:
env = os.environ.copy(); env["PYTHONPATH"] = str(CODE_DIR / "src"); env["PYTHONUNBUFFERED"] = "1"
def run_stream(command, cwd=CODE_DIR, expect_success=True):
    started = time.monotonic(); process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout: lines.append(line); print(line, end="", flush=True)
    code_returned = process.wait()
    if expect_success and code_returned: raise subprocess.CalledProcessError(code_returned, command)
    if not expect_success and not code_returned: raise AssertionError("command unexpectedly succeeded")
    return time.monotonic() - started, "".join(lines)
_, targeted_output = run_stream([sys.executable, "-u", "-m", "unittest", "-v", "tests.test_panderm_blockers", "tests.test_panderm_base_c1_finetune", "tests.test_panderm_notebooks"])
_, suite_output = run_stream([sys.executable, "-u", "-m", "unittest", "discover", "-s", "tests", "-v"])
_, smoke_output = run_stream([sys.executable, "-u", "scripts/smoke_test.py"])
targeted_checks = {"targeted_ok": "OK" in targeted_output, "suite_ok": "OK" in suite_output, "smoke_ok": "OK" in smoke_output or "PASS" in smoke_output.upper()}
assert all(targeted_checks.values()), targeted_checks
# CLI must refuse illegal combinations before any weight is touched. base_cli is
# itself legal (warm-up <= epochs), so each case below fails for its own reason
# rather than being masked by a shared error.
illegal = [
    ["--variant", "C4"],
    ["--generated-manifest", "/tmp/nope.csv"],
    ["--run-version", "v2_other"],
    ["--df-target-count", "586"],
    ["--accumulation-steps", "0"],
    ["--batch-size", "0"],
    ["--warmup-epochs", "999"],
    ["--evaluation-scope", "full"],
    ["--drop-path", "0.3"],
    ["--no-amp"],
    ["--seed", "1"],
    ["--epochs", "50"],
]
base_cli = [sys.executable, "-u", "-m", "ddpm_derm.train_panderm", "--seed", "0", "--epochs", "5", "--warmup-epochs", "5", "--checkpoint", str(CHECKPOINT_PATH), "--upstream-dir", str(UPSTREAM_DIR), "--output-dir", "/content/panderm-illegal"]
_, base_legal_output = run_stream(base_cli + ["--checkpoint-sha256", "0" * 64], expect_success=False)
assert "SHA-256 mismatch" in base_legal_output, "base CLI must reach the checkpoint gate, so each illegal case below fails for its own reason"
for extra in illegal:
    run_stream(base_cli + extra, expect_success=False)
print("CLI illegal combinations rejected:", len(illegal))

## Phase 3 CHECK - real weight load, full trainability, single-batch forward/backward with backbone gradients

In [ ]:
from PIL import Image
from ddpm_derm import panderm
train_transform = panderm.build_train_transform()
eval_transform = panderm.build_eval_transform()
model = panderm.build_panderm_classifier(checkpoint_path=CHECKPOINT_PATH, upstream_dir=UPSTREAM_DIR, drop_path=DROP_PATH).cuda()
backbone_count = panderm.assert_full_trainability(model)
total_params, trainable_params = panderm.parameter_counts(model)
assert total_params == trainable_params, "full fine-tuning requires every parameter trainable"
assert model.input_resolution == (224, 224) and model.freeze_mode == "full_finetune"
assert model.head.out_features == 7 and model.head.in_features == 768
sample_paths = [LOCAL_DATA_DIR / path for path in frames["train"]["image_path"].head(4)]
batch = torch.stack([eval_transform(Image.open(path).convert("RGB")) for path in sample_paths]).cuda()
assert tuple(batch.shape) == (4, 3, 224, 224)
model.train(); logits = model(batch)
assert tuple(logits.shape) == (4, 7) and torch.isfinite(logits).all()
optimizer = panderm.build_optimizer(model, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, layer_decay=LAYER_DECAY)
covered = panderm.verify_optimizer_covers_parameters_once(optimizer, model)
assert covered == sum(1 for p in model.parameters() if p.requires_grad)
schedule = panderm.WarmupCosineSchedule(optimizer, base_lr=LEARNING_RATE, warmup_epochs=WARMUP_EPOCHS, epochs=VALIDATION_EPOCHS if VALIDATION_EPOCHS >= WARMUP_EPOCHS else WARMUP_EPOCHS, steps_per_epoch=4)
scaler = torch.amp.GradScaler("cuda", enabled=True)
before_parameters = panderm.snapshot_parameters(model)
targets = torch.tensor([0, 3, 5, 6], device="cuda")
criterion = torch.nn.CrossEntropyLoss()
model.zero_grad(set_to_none=True)
for micro in range(ACCUMULATION_STEPS):
    with torch.autocast(device_type="cuda", enabled=True):
        loss = criterion(model(batch), targets)
    assert torch.isfinite(loss)
    scaler.scale(loss * panderm.accumulation_loss_scale(micro, ACCUMULATION_STEPS, ACCUMULATION_STEPS)).backward()
gradient_report = panderm.backbone_gradient_report(model)
assert gradient_report["all_backbone_parameters_have_gradient"], "backbone must receive gradients; this is not a linear probe"
assert gradient_report["backbone_gradients_finite"] and gradient_report["head_parameters_have_gradient"]
scaler.unscale_(optimizer); panderm.require_finite_gradients(model.parameters())
scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); schedule.step()
changed_backbone = panderm.changed_parameter_count(before_parameters, model)
assert changed_backbone > 0, "backbone parameters must actually update"
backbone_gradient_verified = bool(gradient_report["all_backbone_parameters_have_gradient"])
backbone_parameters_updated = changed_backbone > 0
model_details = panderm.model_identity(model, train_transform=train_transform, eval_transform=eval_transform, checkpoint_sha256=checkpoint_sha256)
dependency_versions = panderm.dependency_versions()
assert model_details["arch"] == ARCH and model_details["all_parameters_trainable"] is True
assert model_details["embed_dim"] == 768 and model_details["depth"] == 12 and model_details["patch_size"] == 16
assert model_details["normalization_mean"] == [0.485, 0.456, 0.406]
scales = panderm.layer_decay_scales()
assert len(scales) == 14 and abs(scales[13] - 1.0) < 1e-12 and abs(scales[0] - LAYER_DECAY ** 13) < 1e-12
print(json.dumps({"backbone_parameter_count": backbone_count, "total_params": total_params, "changed_backbone_tensors": changed_backbone, "optimizer_groups": len(optimizer.param_groups), "gradient_report": gradient_report, "dependency_versions": dependency_versions}, indent=2))
del model, optimizer, schedule, scaler, batch, logits, before_parameters; torch.cuda.empty_cache()

## Phase 4 RUN - seed 0, five-epoch validation-only run plus resume / mismatch smoke

In [ ]:
import copy
from ddpm_derm import train_panderm
from datetime import datetime, timezone
existing_attempts = sorted(path for path in VALIDATION_RUNS_ROOT.iterdir() if path.is_dir())
assert len(existing_attempts) <= 1, f"only one validation attempt directory is permitted: {existing_attempts}"
if existing_attempts:
    VALIDATION_DIR = existing_attempts[0]; validation_id = VALIDATION_DIR.name
else:
    validation_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    VALIDATION_DIR = panderm_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_RUNS_ROOT / validation_id).relative_to(SHARED_RUN_ROOT))
GATE_ROOT = panderm_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_DIR / "non_collapse_gate").relative_to(SHARED_RUN_ROOT))
formal_output_identity = f"{sentinel['shared_root_uuid']}:{RUN_VERSION}:formal"
gate_command = [sys.executable, "-u", "-m", "ddpm_derm.train_panderm", "--variant", VARIANT, "--seed", "0", "--epochs", str(VALIDATION_EPOCHS), "--batch-size", str(BATCH_SIZE), "--accumulation-steps", str(ACCUMULATION_STEPS), "--lr", str(LEARNING_RATE), "--weight-decay", str(WEIGHT_DECAY), "--warmup-epochs", str(VALIDATION_EPOCHS), "--layer-decay", str(LAYER_DECAY), "--df-target-count", str(DF_TARGET_COUNT), "--checkpoint", str(CHECKPOINT_PATH), "--checkpoint-sha256", checkpoint_sha256, "--upstream-dir", str(UPSTREAM_DIR), "--upstream-commit", upstream_commit, "--evaluation-scope", "validation_only", "--output-dir", str(GATE_ROOT), "--run-version", RUN_VERSION, "--shared-root-uuid", sentinel["shared_root_uuid"], "--formal-output-identity", formal_output_identity, "--fixed-split-identity", fixed_split_identity]
checkpoint_dir = GATE_ROOT / "checkpoints" / ARCH / f"{VARIANT}_seed0"
result_path = GATE_ROOT / "results" / ARCH / f"results_{VARIANT}_seed0.json"
if (checkpoint_dir / "last.pt").is_file(): gate_command.append("--resume")
gate_seconds, gate_output = run_stream(gate_command)
assert "[test]" not in gate_output and "checkpoint_saved=last.pt" in gate_output
result = json.loads(result_path.read_text(encoding="utf-8"))
assert result["evaluation_scope"] == "validation_only" and result["test_metrics"] is None
assert result["data_counts"] == {"train": 7495, "val": 1510, "test": None}
assert len(result["history"]) == VALIDATION_EPOCHS
best_path, last_path = checkpoint_dir / "best.pt", checkpoint_dir / "last.pt"
assert best_path.is_file() and last_path.is_file()
for path in (best_path, last_path):
    reopened = train_panderm.load_checkpoint_safe(path, map_location="cpu", expected_identity=result["run_identity"], expected_result_checkpoint=result["checkpoint_integrity"][path.name])
    assert reopened["checkpoint_format"] == CHECKPOINT_FORMAT
    assert "model_state_dict" in reopened and "head_state_dict" not in reopened
    assert set(reopened) >= {"optimizer_state_dict", "scheduler_state_dict", "scaler_state_dict", "rng_state", "run_identity"}
    assert reopened["run_identity"] == result["run_identity"]
    panderm_run.require_matching_identity(reopened["run_identity"], result["run_identity"])
assert train_panderm.load_checkpoint_safe(last_path, map_location="cpu", expected_identity=result["run_identity"], expected_result_checkpoint=result["checkpoint_integrity"]["last.pt"])["epoch"] == VALIDATION_EPOCHS
best_checkpoint, last_checkpoint = train_panderm.load_completed_checkpoint_pair_safe(best_path=best_path, last_path=last_path, result=result, model=None, expected_identity=result["run_identity"], map_location="cpu")
panderm_run.require_completed_artifact_identities(expected=result["run_identity"], result=result, best_checkpoint=best_checkpoint, last_checkpoint=last_checkpoint)
before_state = (sha256(last_path), last_path.stat().st_mtime_ns)
_, resumed_output = run_stream(gate_command + ["--resume"]); assert "[resume]" in resumed_output and "[skip]" in resumed_output
for index, replacement in (("--layer-decay", "0.75"), ("--lr", "1e-4"), ("--weight-decay", "0.01"), ("--accumulation-steps", "4"), ("--batch-size", "8")):
    mismatch = gate_command.copy(); mismatch[mismatch.index(index) + 1] = replacement
    run_stream(mismatch + ["--resume"], expect_success=False)
saved_identity = last_checkpoint["run_identity"]
for key, value in (("checkpoint_sha256", "0" * 64), ("upstream_commit", "1" * 40), ("claim_boundary", "confirmed"), ("evaluation_scope", "full")):
    drifted = copy.deepcopy(saved_identity); drifted[key] = value
    try: panderm_run.require_matching_identity(saved_identity, drifted); raise AssertionError(f"identity drift accepted: {key}")
    except ValueError: pass
truncated = {key: value for key, value in saved_identity.items() if key != "objective"}
try: panderm_run.require_expected_identity_complete(truncated); raise AssertionError("truncated expectation accepted")
except ValueError: pass
assert (sha256(last_path), last_path.stat().st_mtime_ns) == before_state, "a rejected resume must not mutate the checkpoint"
child_code = "import subprocess,sys; subprocess.run(sys.argv[1:], check=True)"
subprocess.run([sys.executable, "-c", child_code] + gate_command + ["--resume"], cwd=CODE_DIR, env=env, check=True)
resume_checks = {"resume": True, "mismatches_rejected_without_mutation": True, "child_process_drive_only_restore": True}
print(json.dumps({"gate_seconds": gate_seconds, "best_val_df_f1": result["best_val_df_f1"], "resume_checks": resume_checks}, indent=2))

## Phase 5 RUN - non-collapse gate (writes a failure record and stops before formal training)

In [ ]:
predicted = torch.tensor(result["validation_metrics"]["confusion_matrix"]).sum(dim=0)
prediction_counts = {name: int(predicted[index]) for index, name in enumerate(config.CLASS_NAMES)}
identity_complete = set(result["run_identity"]) == set(panderm_run.IMMUTABLE_IDENTITY_KEYS)
trainer_source = (CODE_DIR / "src" / "ddpm_derm" / "train_panderm.py").read_text(encoding="utf-8")
validation_notebook = json.loads((CODE_DIR / "notebooks" / "colab_panderm_base_c1_finetune_validation.ipynb").read_text(encoding="utf-8"))
validation_code = "\n".join("".join(cell.get("source", [])) for cell in validation_notebook["cells"] if cell["cell_type"] == "code")
panderm_run.validate_validation_notebook_source(validation_code)
formal_notebook = json.loads((CODE_DIR / "notebooks" / "colab_panderm_base_c1_finetune_classifier.ipynb").read_text(encoding="utf-8"))
formal_code = "\n".join("".join(cell.get("source", [])) for cell in formal_notebook["cells"] if cell["cell_type"] == "code")
source_guards = {
    "trainer_has_no_test_split_load": 'load_split("test")' not in trainer_source,
    "trainer_has_no_full_scope": 'evaluation_scope == "full"' not in trainer_source,
    "validation_notebook_uses_allowlist_staging": "panderm_run.stage_validation_data" in validation_code,
    "validation_notebook_copy_ast_safe": True,
    "formal_notebook_has_no_test_split_load": 'load_split("test")' not in formal_code,
    "formal_notebook_has_no_training_entrypoint": "ddpm_derm.train_panderm" not in formal_code,
}
test_access_probes = {}
for scenario in ("before_validation_match", "before_three_seed_completion", "forged_validation_pass"):
    try:
        panderm_run.require_provenance_clearance(
            upstream_commit=upstream_commit,
            checkpoint_sha256=checkpoint_sha256,
            purpose=panderm_run.TEST_ACCESS,
        )
    except ValueError as error:
        test_access_probes[scenario] = str(error) == panderm_run.PROHIBITED_FORMAL_TEST_REASON
    else:
        test_access_probes[scenario] = False
no_test_access = all(source_guards.values()) and all(test_access_probes.values()) and result["test_metrics"] is None and result["data_counts"]["test"] is None
checks = panderm_run.evaluate_non_collapse_gate(result=result, prediction_counts=prediction_counts, backbone_gradient_verified=backbone_gradient_verified, backbone_parameters_updated=backbone_parameters_updated, identity_complete=identity_complete, no_test_access=no_test_access, provenance_allows_next_stage=bool(provenance_clearance))
assert set(checks) == set(panderm_run.NON_COLLAPSE_CHECK_KEYS), "non-collapse gate must evaluate every fixed check"
for required_check in ("finite_losses", "finite_metrics", "best_validation_df_f1_positive", "predicted_df_positive", "at_least_two_predicted_classes", "not_all_nv", "not_all_df", "backbone_gradient_verified", "backbone_parameters_updated", "identity_complete", "test_metrics_null", "no_test_access", "provenance_allows_next_stage"):
    assert required_check in checks, f"non-collapse gate is missing {required_check}"
gate_metrics = {"best_validation_df_f1": result["best_val_df_f1"], "best_epoch": max(result["history"], key=lambda item: item["val_df_f1"])["epoch"], "history": result["history"], "prediction_counts": prediction_counts, "checks": checks, "test_access_probes": test_access_probes, "source_guards": source_guards, "elapsed_seconds": gate_seconds, "changed_backbone_tensors": changed_backbone, "gradient_report": gradient_report}
gate_failures = [key for key, value in checks.items() if not value]
if gate_failures:
    failure_guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
    assert failure_guard_after == before_guard, "read-only guard files changed"
    failure_record = {"validation_status": "VALIDATION FAILED", "formal_training_started": False, "formal_training_allowed": False, "test_access_allowed": False, "run_version": RUN_VERSION, "git_commit": commit, "upstream_commit": upstream_commit, "checkpoint_sha256": checkpoint_sha256, "arch": ARCH, "variant": VARIANT, "gate_failures": gate_failures, "non_collapse_gate": gate_metrics, "model_identity": model_details, "fixed_split_identity": fixed_split_identity, "manifest_sha256": manifest_sha256, "shared_root_uuid": sentinel["shared_root_uuid"], "provenance": provenance, "claim_boundary": panderm_run.CLAIM_BOUNDARY, "evaluation_scope": "validation_only", "test_metrics": None, "validation_artifact_directory": str(VALIDATION_DIR)}
    panderm_run.write_json_atomic(VALIDATION_DIR / "validation_failure.json", failure_record)
    panderm_run.write_json_atomic(LATEST_FAILURE_RECORD, failure_record)
    assert not VALIDATION_RECORD.exists(), "a failed gate must never leave a success validation_record"
    print("VALIDATION FAILED")
    print("formal_training_allowed=false")
    print("test_access_allowed=false")
    print("run_version=v1_panderm_base_c1_finetune")
    raise RuntimeError(gate_failures)
print(json.dumps(checks, indent=2))


## Phase 6 REVIEW - immutable validation record and hard stop before formal training

In [ ]:
guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
assert guard_after == before_guard, "read-only guard files changed"
panderm_run.require_no_deployment_contamination(CODE_DIR)
record = {"validation_status": "VALIDATION PASSED", "formal_training_started": False, "formal_training_allowed": False, "test_access_allowed": False, "run_version": RUN_VERSION, "arch": ARCH, "variant": VARIANT, "git_commit": commit, "upstream_repo": PANDERM_UPSTREAM_REPO, "upstream_commit": upstream_commit, "checkpoint_filename": PANDERM_CHECKPOINT_FILENAME, "checkpoint_source_url": panderm_run.CHECKPOINT_SOURCE_URL, "checkpoint_sha256": checkpoint_sha256, "checkpoint_sha256_provenance": panderm_run.CHECKPOINT_SHA256_PROVENANCE, "checkpoint_bytes": CHECKPOINT_PATH.stat().st_size, "checkpoint_format": CHECKPOINT_FORMAT, "model_identity": model_details, "dependency_versions": dependency_versions, "fixed_split_identity": fixed_split_identity, "manifest_sha256": manifest_sha256, "c1_construction": {"strategy": "duplicate_real_train_df", "df_target_count": DF_TARGET_COUNT, "synthetic_images_used": False, "class_counts": c1_counts, "train_rows": len(c1_frame)}, "objective": result["run_identity"]["objective"], "optimization": result["run_identity"]["optimization"], "shared_root_uuid": sentinel["shared_root_uuid"], "shared_root_identity": sentinel, "formal_output_identity": formal_output_identity, "evaluation_scope": "validation_only", "non_collapse_gate": gate_metrics, "resume_mismatch_checks": resume_checks, "targeted_check_results": targeted_checks, "drive_probes": drive_probe, "provenance": provenance, "license_review": provenance["license_review"], "contamination_review": provenance["contamination_review"], "claim_boundary": panderm_run.CLAIM_BOUNDARY, "results_grade": "exploratory", "deployment_allowed": False, "attribution": panderm_run.ATTRIBUTION, "validation_artifact_directory": str(VALIDATION_DIR), "existing_artifacts_unchanged": True, "gh_token_present": GH_TOKEN_PRESENT, "test_metrics": None}
panderm_run.write_json_atomic(VALIDATION_DIR / "validation_record.json", record)
panderm_run.write_json_atomic(VALIDATION_RECORD, record)
assert not FORMAL_ROOT.exists(), "PanDerm v1 must never create formal artifacts"
print(json.dumps(record, indent=2))
print("VALIDATION PASSED")
print("formal_training_allowed=false")
print("test_access_allowed=false")
print("run_version=v1_panderm_base_c1_finetune")
print("claim_boundary=suggestive_exploratory_only")
